In [ ]:
import csv
import json
from pathlib import Path

import ijson
import numpy as np

archivos = [
    "../data/interim/[NOMBRE_ARCHIVO].json"
]

Path("../data/processed/").mkdir(exist_ok=True)

for archivo in archivos:
    csv_path = Path("../data/processed/") / (Path(archivo).stem + ".csv")

    validos = 0
    descartados = 0
    incompleto = False

    with open(csv_path, "w", newline="", encoding="utf-8") as salida:
        writer = csv.writer(salida)
        writer.writerow(["Event", "Coda", "Type", "Data"])

        try:
            with open(archivo, "rb") as entrada:
                for evento in ijson.items(entrada, "T.item"):
                    if not isinstance(evento, dict):
                        descartados += 1
                        continue

                    if evento.get("Voting_Detection") != "Confirmed":
                        continue

                    try:
                        data_bhz = evento["BREF"][0]["BHZ"][0]["Data"]
                    except (KeyError, IndexError, TypeError):
                        descartados += 1
                        continue

                    if not isinstance(data_bhz, list) or not data_bhz:
                        descartados += 1
                        continue

                    try:
                        signal_np = np.asarray(data_bhz, dtype=float)

                        if not np.isfinite(signal_np).all():
                            descartados += 1
                            continue
                    except (TypeError, ValueError):
                        descartados += 1
                        continue

                    writer.writerow([
                        evento.get("Event"),
                        evento.get("Coda"),
                        evento.get("Type"),
                        json.dumps(
                            signal_np.tolist(),
                            separators=(",", ":")
                        ),
                    ])

                    validos += 1

        except ijson.common.IncompleteJSONError:
            incompleto = True

    estado = "TRUNCADO" if incompleto else "COMPLETO"

    print(f"{archivo}: {estado}")
    print(f"  Registros validos: {validos}")
    print(f"  Registros descartados: {descartados}")
    print(f"  CSV generado: {csv_path}")

data_proccessed/TREM_IGEPN_1.json: TRUNCADO
  Registros validos: 8
  Registros descartados: 14
  CSV generado: data\TREM_IGEPN_1.csv
data_proccessed/TREMI_IGEPN_1.json: TRUNCADO
  Registros validos: 59
  Registros descartados: 35
  CSV generado: data\TREMI_IGEPN_1.csv
